# ML API Deployment Notebook

This notebook deploys the ML Models API on Google Colab with ngrok tunneling.

**Features:**
- Recommendation System (Popularity, Collaborative, Content-based)
- Image Similarity Search & Classification
- Arabic Chatbot (Llama 3 8B with LoRA)

**Requirements:**
- Google Colab with GPU runtime (T4 or better)
- ngrok account (free tier works)
- HuggingFace account with Llama 3 access (for chatbot)

## 1. Check GPU and Runtime

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Install Dependencies

In [ ]:
# Install all required packages
!pip install -q \
    fastapi==0.109.0 \
    uvicorn[standard]==0.27.0 \
    pyngrok==7.0.0 \
    python-jose[cryptography]==3.3.0 \
    passlib[bcrypt]==1.7.4 \
    python-multipart==0.0.6 \
    sqlalchemy==2.0.25 \
    slowapi==0.1.9 \
    scikit-learn==1.4.0 \
    pillow==10.2.0 \
    joblib==1.3.2 \
    peft==0.7.1 \
    bitsandbytes==0.41.3 \
    accelerate==0.25.0 \
    nest-asyncio==1.6.0

print("Dependencies installed successfully!")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive for persistent storage
import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/ML_API_Project'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/saved_models', exist_ok=True)
print(f"Project directory: {DRIVE_PROJECT_DIR}")

## 4. Configure Secrets

Use Colab's Secrets feature (key icon in left sidebar) to store:
- `NGROK_AUTH_TOKEN`: Your ngrok auth token
- `HF_TOKEN`: Your HuggingFace token (for Llama 3 access)
- `JWT_SECRET`: Secret key for JWT tokens
- `ADMIN_PASSWORD`: Password for admin user

In [ ]:
from google.colab import userdata
import os

# Load secrets from Colab Secrets
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    print("NGROK_AUTH_TOKEN loaded from secrets")
except:
    NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ")

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from secrets")
except:
    HF_TOKEN = input("Enter your HuggingFace token (or press Enter to skip chatbot): ") or None

try:
    JWT_SECRET = userdata.get('JWT_SECRET')
    print("JWT_SECRET loaded from secrets")
except:
    import secrets
    JWT_SECRET = secrets.token_urlsafe(32)
    print(f"Generated JWT_SECRET: {JWT_SECRET[:20]}...")

try:
    ADMIN_PASSWORD = userdata.get('ADMIN_PASSWORD')
    print("ADMIN_PASSWORD loaded from secrets")
except:
    ADMIN_PASSWORD = input("Enter admin password (or press Enter for default): ") or "admin123"

# Set environment variables
os.environ['JWT_SECRET_KEY'] = JWT_SECRET
os.environ['HF_TOKEN'] = HF_TOKEN or ''
os.environ['ADMIN_USERNAME'] = 'admin'
os.environ['ADMIN_EMAIL'] = 'admin@api.local'
os.environ['ADMIN_PASSWORD'] = ADMIN_PASSWORD

print("\nEnvironment configured!")

## 5. Setup ngrok

In [ ]:
from pyngrok import ngrok, conf

# Set ngrok auth token
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any existing tunnels
ngrok.kill()

print("ngrok configured successfully!")

## 6. Upload or Clone ML API Code

**Option A:** Upload from local machine (recommended for development)

**Option B:** Clone from GitHub repository

In [ ]:
# Option A: Upload ml_api folder
# Run this cell, then drag and drop your ml_api.zip file

from google.colab import files
import zipfile
import os

print("Upload ml_api.zip (zip of the ml_api folder):")
print("Or skip this cell if you want to use Option B (GitHub clone)")

try:
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content')
            os.remove(filename)
            print(f"Extracted {filename}")
except:
    print("Upload skipped or cancelled")

In [ ]:
# Option B: Clone from GitHub (uncomment and modify URL)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# !mv YOUR_REPO/ml_api /content/ml_api

# Verify ml_api exists
import os
if os.path.exists('/content/ml_api'):
    print("ml_api folder found!")
    !ls -la /content/ml_api
else:
    print("ERROR: ml_api folder not found. Please upload or clone it.")

## 7. Upload Model Files

Upload pre-trained models or copy from Google Drive.

In [ ]:
import os
import shutil

# Create saved_models directories
os.makedirs('/content/ml_api/saved_models/recommendation', exist_ok=True)
os.makedirs('/content/ml_api/saved_models/image', exist_ok=True)
os.makedirs('/content/ml_api/saved_models/chatbot', exist_ok=True)
os.makedirs('/content/ml_api/data', exist_ok=True)

# Copy models from Drive if they exist
DRIVE_MODELS = f'{DRIVE_PROJECT_DIR}/saved_models'

def copy_if_exists(src, dst):
    if os.path.exists(src):
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print(f"Copied: {src}")
        return True
    return False

# Try to copy from Drive
copied = False
copied |= copy_if_exists(f'{DRIVE_MODELS}/recommendation', '/content/ml_api/saved_models/recommendation')
copied |= copy_if_exists(f'{DRIVE_MODELS}/image', '/content/ml_api/saved_models/image')
copied |= copy_if_exists(f'{DRIVE_MODELS}/chatbot', '/content/ml_api/saved_models/chatbot')

if not copied:
    print("No models found in Drive. You can upload them manually or run the notebooks to generate them.")

print("\nModel directories:")
!ls -la /content/ml_api/saved_models/

In [ ]:
# Upload model files manually if needed
# Uncomment and run this cell to upload specific model files

# from google.colab import files
# print("Upload model files (pkl, npy, keras):")
# uploaded = files.upload()
# 
# for filename in uploaded.keys():
#     # Move to appropriate directory based on filename
#     if 'popularity' in filename or 'svd' in filename or 'tfidf' in filename or 'kmeans' in filename:
#         dest = f'/content/ml_api/saved_models/recommendation/{filename}'
#     elif 'resnet' in filename or 'butterfly' in filename or 'feature' in filename:
#         dest = f'/content/ml_api/saved_models/image/{filename}'
#     else:
#         dest = f'/content/ml_api/saved_models/{filename}'
#     shutil.move(filename, dest)
#     print(f"Moved {filename} to {dest}")

## 8. Configure Model Paths

In [ ]:
import os

# Set model paths
os.environ['MODELS_DIR'] = '/content/ml_api/saved_models'
os.environ['DATABASE_PATH'] = '/content/ml_api/data/users.db'
os.environ['LORA_ADAPTER_PATH'] = '/content/ml_api/saved_models/chatbot/llama-3-8B-Arabic'
os.environ['LLAMA_BASE_MODEL'] = 'meta-llama/Meta-Llama-3-8B-Instruct'

print("Model paths configured:")
print(f"  MODELS_DIR: {os.environ['MODELS_DIR']}")
print(f"  DATABASE_PATH: {os.environ['DATABASE_PATH']}")
print(f"  LORA_ADAPTER_PATH: {os.environ['LORA_ADAPTER_PATH']}")

## 9. HuggingFace Login (for Chatbot)

In [ ]:
# Login to HuggingFace for Llama 3 access
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace!")
else:
    print("HF_TOKEN not set. Chatbot functionality will be limited.")
    print("To use the chatbot, add your HuggingFace token to Colab Secrets.")

## 10. Start the API Server

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import sys
sys.path.insert(0, '/content')

# Import and configure the app
from ml_api.main import app

print("FastAPI app loaded successfully!")

In [ ]:
import threading
import uvicorn
from pyngrok import ngrok

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"Public URL: {public_url}")
print(f"{'='*60}")
print(f"\nAPI Documentation: {public_url}/docs")
print(f"Health Check: {public_url}/health")
print(f"\nAdmin credentials:")
print(f"  Username: {os.environ.get('ADMIN_USERNAME', 'admin')}")
print(f"  Password: {os.environ.get('ADMIN_PASSWORD', '(check your secrets)')}")
print(f"{'='*60}\n")

# Run server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server started! The cell will keep running to maintain the tunnel.")
print("Press the stop button to shut down the server.")

## 11. Test Endpoints

Run the following cells to test the API endpoints.

In [ ]:
import requests
import json

# Get the public URL
BASE_URL = str(public_url).replace('NgrokTunnel: "', '').replace('" -> "http://localhost:8000"', '')
if 'NgrokTunnel' in BASE_URL:
    # Extract URL from NgrokTunnel object
    BASE_URL = public_url.public_url
print(f"Base URL: {BASE_URL}")

# Test health endpoint
print("\n--- Health Check ---")
response = requests.get(f"{BASE_URL}/health")
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Test authentication
print("--- Login ---")
login_data = {
    "username": os.environ.get('ADMIN_USERNAME', 'admin'),
    "password": os.environ.get('ADMIN_PASSWORD', 'admin123')
}
response = requests.post(f"{BASE_URL}/auth/login", json=login_data)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    tokens = response.json()
    ACCESS_TOKEN = tokens['access_token']
    print(f"Access token: {ACCESS_TOKEN[:50]}...")
    
    # Create headers for authenticated requests
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
else:
    print(f"Login failed: {response.text}")
    headers = {}

In [ ]:
# Test /auth/me endpoint
if headers:
    print("\n--- Current User ---")
    response = requests.get(f"{BASE_URL}/auth/me", headers=headers)
    print(f"Status: {response.status_code}")
    print(json.dumps(response.json(), indent=2))

In [ ]:
# Test recommendation endpoints
if headers:
    print("\n--- Popular Products ---")
    response = requests.get(f"{BASE_URL}/recommend/popular?top_n=5", headers=headers)
    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"Error: {response.text}")

In [ ]:
# Test content-based recommendation
if headers:
    print("\n--- Content-Based Recommendation ---")
    data = {"search_query": "cutting tool", "top_n": 5}
    response = requests.post(f"{BASE_URL}/recommend/content-based", json=data, headers=headers)
    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"Error: {response.text}")

In [ ]:
# Test chatbot status
if headers:
    print("\n--- Chatbot Status ---")
    response = requests.get(f"{BASE_URL}/chat/status", headers=headers)
    print(f"Status: {response.status_code}")
    print(json.dumps(response.json(), indent=2))

In [ ]:
# Test chatbot (this will load the model on first call)
if headers:
    print("\n--- Chatbot Message ---")
    print("Note: First call will load the model (may take a minute)...")
    data = {
        "message": "ما هي عاصمة مصر؟",
        "max_tokens": 100,
        "temperature": 0.4
    }
    response = requests.post(f"{BASE_URL}/chat/message", json=data, headers=headers, timeout=120)
    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        result = response.json()
        print(f"\nQuestion: {result['input_message']}")
        print(f"Response: {result['response']}")
        print(f"\nTokens: {result['tokens_generated']}, Time: {result['generation_time_ms']}ms")
    else:
        print(f"Error: {response.text}")

## 12. Save Models to Drive (Optional)

Save any trained models back to Google Drive for persistence.

In [ ]:
# Save models back to Drive for persistence
import shutil

def save_to_drive(src, dst):
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print(f"Saved: {src} -> {dst}")

# Uncomment to save models
# save_to_drive('/content/ml_api/saved_models/recommendation', f'{DRIVE_MODELS}/recommendation')
# save_to_drive('/content/ml_api/saved_models/image', f'{DRIVE_MODELS}/image')
# save_to_drive('/content/ml_api/saved_models/chatbot', f'{DRIVE_MODELS}/chatbot')
# save_to_drive('/content/ml_api/data/users.db', f'{DRIVE_PROJECT_DIR}/users.db')

print("Run the save commands above to persist models to Drive.")

## 13. Keep Session Alive (Optional)

Run this cell to prevent Colab from timing out.

In [ ]:
# Keep session alive with periodic activity
import time
from IPython.display import clear_output

print("Keeping session alive... Press stop button to end.")
print(f"API URL: {public_url.public_url}")

counter = 0
while True:
    time.sleep(60)  # Wait 1 minute
    counter += 1
    
    # Periodic health check
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=10)
        status = "healthy" if response.status_code == 200 else "unhealthy"
    except:
        status = "unreachable"
    
    clear_output(wait=True)
    print(f"Session active for {counter} minutes")
    print(f"API Status: {status}")
    print(f"API URL: {public_url.public_url}")

## Troubleshooting

### Common Issues

1. **ngrok tunnel not working**
   - Check your ngrok auth token is correct
   - Free tier has connection limits

2. **Models not loading**
   - Ensure model files are in the correct directories
   - Check the health endpoint for model status

3. **Chatbot not working**
   - Verify HuggingFace token has Llama 3 access
   - Check GPU memory (needs ~5GB free)

4. **Session timeout**
   - Use the "Keep Session Alive" cell
   - Consider Colab Pro for longer sessions